# Word-Level LSTM Text Generator

This notebook demonstrates how to create a word-level LSTM text generator using a book as the dataset. We will download a book, preprocess the text, build and train an LSTM model, and finally generate text based on the trained model.

In [1]:
import requests
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt

def download_book(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    else:
        raise Exception('Failed to download the book.')

In [2]:
# Download a book from Project Gutenberg
book_url = 'https://www.gutenberg.org/files/11/11-0.txt'  # Alice's Adventures in Wonderland
book_text = download_book(book_url)
print(book_text[:500])  # Print the first 500 characters of the book

In [3]:
def preprocess_text(text):
    text = text.lower()
    text = text.replace('\n', ' ')
    return text

processed_text = preprocess_text(book_text)
print(processed_text[:500])  # Print the first 500 characters of the processed text

In [4]:
def create_sequences(text, seq_length):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts([text])
    total_words = len(tokenizer.word_index) + 1
    input_sequences = []
    for i in range(seq_length, len(text.split())):
        seq = text.split()[i-seq_length:i]
        input_sequences.append(' '.join(seq))
    input_sequences = tokenizer.texts_to_sequences(input_sequences)
    input_sequences = np.array(input_sequences)
    return input_sequences, total_words

seq_length = 10
input_sequences, total_words = create_sequences(processed_text, seq_length)
print(input_sequences[:5])  # Print the first 5 input sequences

In [5]:
def prepare_data(input_sequences):
    X, y = input_sequences[:, :-1], input_sequences[:, -1]
    y = tf.keras.utils.to_categorical(y, num_classes=total_words)
    return X, y

X, y = prepare_data(input_sequences)
print(X.shape, y.shape)  # Print the shapes of X and y

In [6]:
def build_model(total_words, seq_length):
    model = Sequential()
    model.add(Embedding(total_words, 100, input_length=seq_length))
    model.add(LSTM(150, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(150))
    model.add(Dropout(0.2))
    model.add(Dense(total_words, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

model = build_model(total_words, seq_length)
model.summary()  # Print the model summary

In [7]:
history = model.fit(X, y, epochs=100, verbose=1)
plt.plot(history.history['accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.show()

In [8]:
def generate_text(model, tokenizer, seq_length, seed_text, num_words):
    output_text = seed_text
    for _ in range(num_words):
        input_seq = tokenizer.texts_to_sequences([output_text])[0]
        input_seq = pad_sequences([input_seq], maxlen=seq_length, padding='pre')
        predicted_word_index = np.argmax(model.predict(input_seq), axis=-1)
        output_word = tokenizer.index_word[predicted_word_index[0]]
        output_text += ' ' + output_word
    return output_text

seed_text = 'alice was'
generated_text = generate_text(model, tokenizer, seq_length, seed_text, 50)
print(generated_text)  # Print the generated text